In [40]:
import yaml
from jinja2 import Template
from langsmith import Client

from dotenv import load_dotenv

In [41]:
load_dotenv()

True

### RAG Pipeline

In [3]:
def build_prompt(preprocessed_context, question):
    prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- You need to answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- As an output you need to provide:

* The answer to the question based on the provided context.
* The list of the IDs of the chunks that were used to answer the question. Only return the ones that are used in the answer.
* Short description (1-2 sentences) of the item based on the description provided in the context.

- The short description should have the name of the item.
- The answer to the question should contain detailed information about the product and returned with detailed specification in bullet points.

Context:
{preprocessed_context}

Question:
{question}
"""

    return prompt

In [8]:
preprocessed_context ="- a \n- b"
question = "What is a?"

In [9]:
prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- You need to answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- As an output you need to provide:

* The answer to the question based on the provided context.
* The list of the IDs of the chunks that were used to answer the question. Only return the ones that are used in the answer.
* Short description (1-2 sentences) of the item based on the description provided in the context.

- The short description should have the name of the item.
- The answer to the question should contain detailed information about the product and returned with detailed specification in bullet points.

Context:
{preprocessed_context}

Question:
{question}
"""

In [10]:
print(prompt)


You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- You need to answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- As an output you need to provide:

* The answer to the question based on the provided context.
* The list of the IDs of the chunks that were used to answer the question. Only return the ones that are used in the answer.
* Short description (1-2 sentences) of the item based on the description provided in the context.

- The short description should have the name of the item.
- The answer to the question should contain detailed information about the product and returned with detailed specification in bullet points.

Context:
- a 
- b

Question:
What is a?



### Jinja Template

In [19]:
jinja_template = """
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- You need to answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- As an output you need to provide:

* The answer to the question based on the provided context.
* The list of the IDs of the chunks that were used to answer the question. Only return the ones that are used in the answer.
* Short description (1-2 sentences) of the item based on the description provided in the context.

- The short description should have the name of the item.
- The answer to the question should contain detailed information about the product and returned with detailed specification in bullet points.

Context:
{{ preprocessed_context }}

Question:
{{ question }}
"""

In [20]:
template = Template(jinja_template)

In [21]:
rendered_template = template.render(preprocessed_context=preprocessed_context, question=question)

In [22]:
print(rendered_template)


You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- You need to answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- As an output you need to provide:

* The answer to the question based on the provided context.
* The list of the IDs of the chunks that were used to answer the question. Only return the ones that are used in the answer.
* Short description (1-2 sentences) of the item based on the description provided in the context.

- The short description should have the name of the item.
- The answer to the question should contain detailed information about the product and returned with detailed specification in bullet points.

Context:
- a 
- b

Question:
What is a?


In [24]:
def build_prompt_jinja(preprocessed_context, question):
    
    prompt = """
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- You need to answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- As an output you need to provide:

* The answer to the question based on the provided context.
* The list of the IDs of the chunks that were used to answer the question. Only return the ones that are used in the answer.
* Short description (1-2 sentences) of the item based on the description provided in the context.

- The short description should have the name of the item.
- The answer to the question should contain detailed information about the product and returned with detailed specification in bullet points.

Context:
{{ preprocessed_context }}

Question:
{{ question }}
"""

    template = Template(prompt)
    rendered_template = template.render(
        preprocessed_context=preprocessed_context, 
        question=question
    )

    return rendered_template

In [25]:
print(build_prompt_jinja(preprocessed_context, question))


You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- You need to answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- As an output you need to provide:

* The answer to the question based on the provided context.
* The list of the IDs of the chunks that were used to answer the question. Only return the ones that are used in the answer.
* Short description (1-2 sentences) of the item based on the description provided in the context.

- The short description should have the name of the item.
- The answer to the question should contain detailed information about the product and returned with detailed specification in bullet points.

Context:
- a 
- b

Question:
What is a?


In [26]:
print(build_prompt_jinja("- some items", "- my silly question"))


You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- You need to answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- As an output you need to provide:

* The answer to the question based on the provided context.
* The list of the IDs of the chunks that were used to answer the question. Only return the ones that are used in the answer.
* Short description (1-2 sentences) of the item based on the description provided in the context.

- The short description should have the name of the item.
- The answer to the question should contain detailed information about the product and returned with detailed specification in bullet points.

Context:
- some items

Question:
- my silly question


### Save prompt template in .yaml file

### Read prompt from .yaml file

In [27]:
def prompt_template_config(yaml_file, prompt_key):
    with open(yaml_file, 'r') as file:
        config = yaml.safe_load(file)
    
    template_content = config['prompts'][prompt_key]
    
    template = Template(template_content)
    
    return template

In [28]:
def build_prompt_jinja(preprocessed_context, question):
    
    template = prompt_template_config("prompts/retrieval_generation.yaml", "retrieval_generation")
    
    rendered_template = template.render(
        preprocessed_context=preprocessed_context, 
        question=question
    )

    return rendered_template

In [31]:
print(build_prompt_jinja(preprocessed_context, question))


You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- You need to answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- As an output you need to provide:

* The answer to the question based on the provided context.
* The list of the IDs of the chunks that were used to answer the question. Only return the ones that are used in the answer.
* Short description (1-2 sentences) of the item based on the description provided in the context.

- The short description should have the name of the item.
- The answer to the question should contain detailed information about the product and returned with detailed specification in bullet points.

Context:
- a 
- b

Question:
What is a?


### Prompt Registries

In [43]:
ls_client = Client()

In [44]:
ls_template = ls_client.pull_prompt("retrieval-generation")

In [45]:
ls_template

ChatPromptTemplate(input_variables=[], input_types={}, partial_variables={}, metadata={'lc_hub_owner': '-', 'lc_hub_repo': 'retrieval-generation', 'lc_hub_commit_hash': 'f4dcb6edfa15bf820d5d50ab2cc9caf2e00c927e7de637b35c49ead12e7d13c6'}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a shopping assistant that can answer questions about the products in stock.\n\nYou will be given a question and a list of context.\n\nInstructions:\n- You need to answer the question based on the provided context only.\n- Never use word context and refer to it as the available products.\n- As an output you need to provide:\n\n* The answer to the question based on the provided context.\n* The list of the IDs of the chunks that were used to answer the question. Only return the ones that are used in the answer.\n* Short description (1-2 sentences) of the item based on the description provided in the context.\n\n- The shor

In [46]:
print(ls_template.messages[0].prompt.template)

You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- You need to answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- As an output you need to provide:

* The answer to the question based on the provided context.
* The list of the IDs of the chunks that were used to answer the question. Only return the ones that are used in the answer.
* Short description (1-2 sentences) of the item based on the description provided in the context.

- The short description should have the name of the item.
- The answer to the question should contain detailed information about the product and returned with detailed specification in bullet points.

Context:
{{ preprocessed_context }}

Question:
{{ question }}


In [47]:
def prompt_template_registry(prompt_name):

    template_content = ls_client.pull_prompt(prompt_name).messages[0].prompt.template

    template = Template(template_content)
    
    return template

In [48]:
print(prompt_template_registry("retrieval-generation").render(preprocessed_context=preprocessed_context, question=question))

You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- You need to answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- As an output you need to provide:

* The answer to the question based on the provided context.
* The list of the IDs of the chunks that were used to answer the question. Only return the ones that are used in the answer.
* Short description (1-2 sentences) of the item based on the description provided in the context.

- The short description should have the name of the item.
- The answer to the question should contain detailed information about the product and returned with detailed specification in bullet points.

Context:
- a 
- b

Question:
What is a?
